MODEL Training 

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [2]:
# 1. Load the Monthly Master Grid (The one we made before the R-script)
# Note: You may need to quickly re-run the Step 1 Master Merge code to generate this 
# if you didn't save it as a CSV earlier. Assuming you have it saved as df_monthly:
# df_monthly = pd.read_csv('Master_Monthly_Grid.csv')

In [3]:
# For now, let's load the two key pieces you definitely have:
df_nfa = pd.read_csv('NFA_Monthly_Triangulated.csv')
df_debt = pd.read_csv('External_Debt_Final_Cleaned.csv')

In [4]:
df_nfa['Date'] = pd.to_datetime(df_nfa['Date'])
df_debt['Date'] = pd.to_datetime(df_debt['Date'])

In [5]:
print(f"NFA Shape: {df_nfa.shape}")
display(df_nfa.head())

NFA Shape: (27948, 3)


,Date,NFA_Triangulated,COUNTRY
0,2001-01-01,81705.744413,Albania
1,2001-02-01,78525.474605,Albania
2,2001-03-01,78759.299676,Albania
3,2001-04-01,83495.331081,Albania
4,2001-05-01,84259.565081,Albania


In [6]:
#PHASE 2: MASTER ML GRID ASSEMBLY & LAG ENGINEERING

In [7]:
# 1. LOAD ALL BASE DATA 
df_exrate = pd.read_csv('exchange_rates_cleaned.csv')
df_cpi = pd.read_csv('CPI_melted.csv') 
df_vix = pd.read_csv('VIX_Final_Cleaned.csv')
df_fed = pd.read_csv('FEDFUNDS_Final_Cleaned.csv')
df_trade = pd.read_csv('international_trade_in_Goods_cleaned.csv')
df_oil = pd.read_csv('Global_price_of_Brent_Crude.csv')

In [8]:
print(f"Exchange Rate Shape & head: {df_exrate.shape}, {df_exrate.head()}")
print(f"CPI Shape & head: {df_cpi.shape}, {df_cpi.head()}")
print(f"VIX Shape & head: {df_vix.shape}, {df_vix.head()}")
print(f"FEDFUNDS Shape & head: {df_fed.shape}, {df_fed.head()}")
print(f"Trade Shape & head: {df_trade.shape}, {df_trade.head()}")
print(f"Oil Shape & head: {df_oil.shape}, {df_oil.head()}")
print(f"NFA Shape & head: {df_nfa.shape}, {df_nfa.head()}")
print(f"Debt Shape & head: {df_debt.shape}, {df_debt.head()}")

Exchange Rate Shape & head: (127890, 3),                             COUNTRY        Date  Exchange_Rate
0  Afghanistan, Islamic Republic of  2000-01-01      46.791100
1  Afghanistan, Islamic Republic of  2000-01-01      46.795800
2  Afghanistan, Islamic Republic of  2000-02-01      47.505938
3  Afghanistan, Islamic Republic of  2000-02-01      47.504800
4  Afghanistan, Islamic Republic of  2000-03-01      47.267200
CPI Shape & head: (9398156, 5),                             COUNTRY  \
0  Afghanistan, Islamic Republic of   
1  Afghanistan, Islamic Republic of   
2  Afghanistan, Islamic Republic of   
3  Afghanistan, Islamic Republic of   
4  Afghanistan, Islamic Republic of   

                                  COICOP_1999                  INDEX_TYPE  \
0  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
1  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
2                               Communication  Consumer price index (CPI)   
3     

In [9]:
print(df_exrate.isnull().sum())
print(df_cpi.isnull().sum())
print(df_vix.isnull().sum())
print(df_fed.isnull().sum())
print(df_trade.isnull().sum())
print(df_oil.isnull().sum())


COUNTRY          0
Date             0
Exchange_Rate    0
dtype: int64
COUNTRY        0
COICOP_1999    0
INDEX_TYPE     0
MONTH_YEAR     0
CPI_VALUE      0
dtype: int64
Date                  0
Monthly_Avg_VIXCLS    0
dtype: int64
FEDFUNDS    0
Date        0
dtype: int64
COUNTRY                   0
INDICATOR                 0
TYPE_OF_TRANSFORMATION    0
Trade_in_Goods            0
Date                      0
dtype: int64
Date               0
Crude_Oil_Price    0
dtype: int64


In [10]:
# 2. STANDARDIZE DATES & RENAME COLUMNS
df_exrate['Date'] = pd.to_datetime(df_exrate['Date'])
if 'MONTH_YEAR' in df_cpi.columns: df_cpi.rename(columns={'MONTH_YEAR': 'Date'}, inplace=True)
df_cpi['Date'] = pd.to_datetime(df_cpi['Date'])
df_vix['Date'] = pd.to_datetime(df_vix['DATE'] if 'DATE' in df_vix.columns else df_vix['Date'])
df_fed['Date'] = pd.to_datetime(df_fed['DATE'] if 'DATE' in df_fed.columns else df_fed['Date'])
df_trade['Date'] = pd.to_datetime(df_trade['Date'] if 'Date' in df_trade.columns else df_trade['Date'])
df_oil['Date'] = pd.to_datetime(df_oil['Date'] if 'Date' in df_oil.columns else df_oil['Date'])
print(df_exrate.columns)
print(df_cpi.columns)
print(df_nfa.columns)
print(df_debt.columns) 
print(df_vix.columns)
print(df_fed.columns)
print(df_oil.columns)
print(df_trade.columns)

Index(['COUNTRY', 'Date', 'Exchange_Rate'], dtype='str')
Index(['COUNTRY', 'COICOP_1999', 'INDEX_TYPE', 'Date', 'CPI_VALUE'], dtype='str')
Index(['Date', 'NFA_Triangulated', 'COUNTRY'], dtype='str')
Index(['Country', 'Series', 'External_Debt', 'Date'], dtype='str')
Index(['Date', 'Monthly_Avg_VIXCLS'], dtype='str')
Index(['FEDFUNDS', 'Date'], dtype='str')
Index(['Date', 'Crude_Oil_Price'], dtype='str')
Index(['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'Trade_in_Goods',
       'Date'],
      dtype='str')


In [11]:
# 3. CPI/EXCHANGE RATE have multiple categories per country per month, so we are going to average them to get a single value per month per country. 
df_exrate = df_exrate.groupby(['COUNTRY', 'Date'], as_index=False)['Exchange_Rate'].mean()
df_cpi = df_cpi.groupby(['COUNTRY', 'Date'], as_index=False)['CPI_VALUE'].mean()

In [12]:
# 3.5 Standardize all Country columns
for df in [df_exrate, df_cpi, df_nfa, df_debt, df_vix, df_fed, df_oil, df_trade]:
    for col in df.columns:
        if col.strip().upper() in ['COUNTRY', 'COUNTRY NAME', 'COUNTRY_NAME']:
            df.rename(columns={col: 'COUNTRY'}, inplace=True)
print(df_exrate.columns)
print(df_cpi.columns)
print(df_nfa.columns)
print(df_debt.columns)
print(df_vix.columns)
print(df_fed.columns)
print(df_oil.columns)
print(df_trade.columns)

Index(['COUNTRY', 'Date', 'Exchange_Rate'], dtype='str')
Index(['COUNTRY', 'Date', 'CPI_VALUE'], dtype='str')
Index(['Date', 'NFA_Triangulated', 'COUNTRY'], dtype='str')
Index(['COUNTRY', 'Series', 'External_Debt', 'Date'], dtype='str')
Index(['Date', 'Monthly_Avg_VIXCLS'], dtype='str')
Index(['FEDFUNDS', 'Date'], dtype='str')
Index(['Date', 'Crude_Oil_Price'], dtype='str')
Index(['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'Trade_in_Goods',
       'Date'],
      dtype='str')


In [13]:
# 4. THE MASTER MERGE
ml_df = df_exrate.copy()
ml_df = pd.merge(ml_df, df_cpi, on=['COUNTRY', 'Date'], how='left')
ml_df = pd.merge(ml_df, df_nfa, on=['COUNTRY', 'Date'], how='left')
ml_df = pd.merge(ml_df, df_debt, on=['COUNTRY', 'Date'], how='left')
ml_df = pd.merge(ml_df, df_trade, on=['COUNTRY', 'Date'], how='left')

In [14]:
# Merge Global Variables (Broadcast to all countries on 'Date')
ml_df = pd.merge(ml_df, df_vix, on='Date', how='left')
ml_df = pd.merge(ml_df, df_fed, on='Date', how='left')
ml_df = pd.merge(ml_df, df_oil, on='Date', how='left')
print(f"Master ML DataFrame Shape: {ml_df.shape}")
display(ml_df.head())

Master ML DataFrame Shape: (302085, 13)


,COUNTRY,Date,Exchange_Rate,CPI_VALUE,NFA_Triangulated,Series,External_Debt,INDICATOR,TYPE_OF_TRANSFORMATION,Trade_in_Goods,Monthly_Avg_VIXCLS,FEDFUNDS,Crude_Oil_Price
0,"Afghanistan, Islamic Republic of",2000-01-01,46.793450,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,23.202000,5.45,25.633333
1,"Afghanistan, Islamic Republic of",2000-02-01,47.505369,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,23.595500,5.73,28.030476
2,"Afghanistan, Islamic Republic of",2000-03-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,22.718261,5.85,27.494348
3,"Afghanistan, Islamic Republic of",2000-04-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,27.164211,6.02,23.153500
4,"Afghanistan, Islamic Republic of",2000-05-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,26.373182,6.27,27.805217


In [15]:
# 5. SORT BY TIME (CRITICAL FOR LAGS)
ml_df = ml_df.sort_values(by=['COUNTRY', 'Date']).reset_index(drop=True)

In [16]:
6. #FEATURE ENGINEERING: LAGS & MOMENTUM (The Engine of the AI)
# 3-Month Lags (Shifted directly by country group)
ml_df['VIX_Lag3'] = ml_df.groupby('COUNTRY')['Monthly_Avg_VIXCLS'].shift(3)
ml_df['FED_Lag3'] = ml_df.groupby('COUNTRY')['FEDFUNDS'].shift(3)

# NFA & CPI Momentum (Percentage change directly by country group)
ml_df['NFA_3M_Change'] = ml_df.groupby('COUNTRY')['NFA_Triangulated'].pct_change(periods=3)
ml_df['CPI_3M_Change'] = ml_df.groupby('COUNTRY')['CPI_VALUE'].pct_change(periods=3)

In [17]:
# External Debt to Reserve Ratio 
ml_df['Debt_to_NFA_Ratio'] = np.where(ml_df['NFA_Triangulated'] > 0, 
                                   ml_df['External_Debt'] / ml_df['NFA_Triangulated'], 
                                   np.nan)

In [18]:
print(f"Final ML Grid Shape: {ml_df.shape}")
display(ml_df[['COUNTRY', 'Date', 'Exchange_Rate', 'NFA_3M_Change', 'Debt_to_NFA_Ratio']].tail(10))

Final ML Grid Shape: (302085, 18)


,COUNTRY,Date,Exchange_Rate,NFA_3M_Change,Debt_to_NFA_Ratio
302075,Zimbabwe,2025-06-01,26.929097,NaN,NaN
302076,Zimbabwe,2025-07-01,26.811515,NaN,NaN
302077,Zimbabwe,2025-08-01,26.760668,NaN,NaN
302078,Zimbabwe,2025-09-01,26.653843,NaN,NaN
302079,Zimbabwe,2025-10-01,26.491161,NaN,NaN
302080,Zimbabwe,2025-11-01,26.258718,NaN,NaN
302081,Zimbabwe,2025-12-01,26.040727,NaN,NaN
302082,Zimbabwe,2026-01-01,25.627562,NaN,NaN
302083,Zimbabwe,2026-02-01,25.698033,NaN,NaN
302084,Zimbabwe,2026-03-01,25.623065,NaN,NaN


In [19]:
ml_df.isnull().sum()

COUNTRY                        0
Date                           0
Exchange_Rate                  0
CPI_VALUE                   7245
NFA_Triangulated          145773
Series                    283651
External_Debt             291329
INDICATOR                  31185
TYPE_OF_TRANSFORMATION     31185
Trade_in_Goods             31185
Monthly_Avg_VIXCLS             0
FEDFUNDS                       0
Crude_Oil_Price                0
VIX_Lag3                     609
FED_Lag3                     609
NFA_3M_Change             146127
CPI_3M_Change               7806
Debt_to_NFA_Ratio         294179
dtype: int64

In [20]:
#ENGINEERING THE TARGET VARIABLE

In [21]:
# 1. Look 3 months into the FUTURE (Notice the -3 instead of 3)
ml_df['Future_Exchange_Rate_3M'] = ml_df.groupby('COUNTRY')['Exchange_Rate'].shift(-3)

In [22]:
# 2. Calculate the percentage change 
# (Positive % means the number went up = local currency lost value to the USD)
ml_df['Exchange_Rate_Change'] = (ml_df['Future_Exchange_Rate_3M'] - ml_df['Exchange_Rate']) / ml_df['Exchange_Rate']

In [23]:
# 3. Create the Binary Classification Target (1 = Crisis, 0 = Safe)
# We define a "Crisis" as a 5% (0.05) devaluation in a 3-month window.
ml_df['Crisis_Target'] = np.where(ml_df['Exchange_Rate_Change'] >= 0.05, 1, 0)

In [24]:
# 4. THE SAFETY VALVE: Masking the Unknown Future
# For the very last 3 months of data (e.g., Oct, Nov, Dec 2024), we don't know the future yet!
# We must force these target rows to be 'NaN'. If we leave them as '0', the AI will 
# learn false information.
ml_df['Crisis_Target'] = np.where(ml_df['Future_Exchange_Rate_3M'].isna(), np.nan, ml_df['Crisis_Target'])
print("Target Variable Created!")

Target Variable Created!


In [25]:
# Let's see how many actual crises our dataset contains!
total_crises = ml_df['Crisis_Target'].sum()
print(f"Total Historical Crises Identified (Target=1): {int(total_crises)}")

Total Historical Crises Identified (Target=1): 5600


In [26]:
# Display the mechanics in action
display(ml_df[['COUNTRY', 'Date', 'Exchange_Rate', 'Future_Exchange_Rate_3M', 'Exchange_Rate_Change', 'Crisis_Target']].head(20))

,COUNTRY,Date,Exchange_Rate,Future_Exchange_Rate_3M,Exchange_Rate_Change,Crisis_Target
0,"Afghanistan, Islamic Republic of",2000-01-01,46.793450,47.267200,0.010124,0.0
1,"Afghanistan, Islamic Republic of",2000-02-01,47.505369,47.267200,-0.005014,0.0
2,"Afghanistan, Islamic Republic of",2000-03-01,47.267200,47.267200,0.000000,0.0
3,"Afghanistan, Islamic Republic of",2000-04-01,47.267200,47.451148,0.003892,0.0
4,"Afghanistan, Islamic Republic of",2000-05-01,47.267200,47.504800,0.005027,0.0
5,"Afghanistan, Islamic Republic of",2000-06-01,47.267200,47.504800,0.005027,0.0
6,"Afghanistan, Islamic Republic of",2000-07-01,47.451148,47.504800,0.001131,0.0
7,"Afghanistan, Islamic Republic of",2000-08-01,47.504800,47.504800,0.000000,0.0
8,"Afghanistan, Islamic Republic of",2000-09-01,47.504800,47.504800,0.000000,0.0
9,"Afghanistan, Islamic Republic of",2000-10-01,47.504800,47.504800,0.000000,0.0


In [27]:
# THE XGBOOST ALGORITHM (TRAINING & TESTING)

In [28]:
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import xgboost as xgb
import numpy as np

In [29]:
# CLEAN THE TARGETS
# XGBoost can handle NaNs in the features, but it CANNOT handle NaNs in the Target variable (Answer Key)!
ml_clean = ml_df.dropna(subset=['Crisis_Target']).copy()

In [30]:
# SELECT THE FEATURES (The Clues)
features = [
    'Monthly_Avg_VIXCLS', 'VIX_Lag3', 
    'FEDFUNDS', 'FED_Lag3', 
    'CPI_VALUE', 'CPI_3M_Change',
    'NFA_Triangulated', 'NFA_3M_Change',
    'External_Debt', 'Debt_to_NFA_Ratio'
]

In [31]:
# STRICT TIME-BASED SPLIT
# Training Data: Learn from history (2000 - 2021)
train_data = ml_clean[ml_clean['Date'].dt.year <= 2021]

In [32]:
# Testing Data: Simulate live trading (2022 - 2024)
test_data = ml_clean[ml_clean['Date'].dt.year > 2021]

X_train = train_data[features]
y_train = train_data['Crisis_Target']

X_test = test_data[features]
y_test = test_data['Crisis_Target']

print(f"Training Rows (2000-2021): {len(X_train)}")
print(f"Testing Rows (2022-2024): {len(X_test)}")

Training Rows (2000-2021): 253176
Testing Rows (2022-2024): 48300


In [33]:
# Calculate the ratio of Safe vs Crisis to balance the AI's attention
ratio = (len(y_train) - y_train.sum()) / y_train.sum()

In [34]:
model = xgb.XGBClassifier(
    n_estimators=300,        # How many decision trees to build
    max_depth=5,             # How deep the trees can think
    learning_rate=0.05,      # How fast it learns (slower prevents overfitting)
    scale_pos_weight=ratio,  # Forces the AI to pay extreme attention to Crises!
    random_state=42,
    eval_metric='auc',
    missing=np.nan           # Explicitly telling XGBoost "Don't panic if you see a NaN"
)
model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [35]:
#PREDICT AND EVALUATE ON THE 2022-2024
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1] # The exact percentage probability of a crash

In [36]:
print(classification_report(y_test, predictions, target_names=['Safe (0)', 'Crisis (1)']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, probabilities):.4f}")

              precision    recall  f1-score   support

    Safe (0)       0.99      0.81      0.89     47413
  Crisis (1)       0.04      0.39      0.07       887

    accuracy                           0.80     48300
   macro avg       0.51      0.60      0.48     48300
weighted avg       0.97      0.80      0.87     48300

ROC-AUC Score: 0.6140
